# Phase 2: confidence-triggered strategy injection

**Kaggle settings:** GPU T4 x2, Internet ON. Steps: calibrate on held-out problems (one GPU, short),
check the calibration, then the main run on both GPUs, merge, analyze.

In [ ]:
!git clone -q https://github.com/Mathindata/seeded-reasoning-lab.git || (cd seeded-reasoning-lab && git pull)
%cd seeded-reasoning-lab
!pip install -q math-verify vllm
import os, sys, json
sys.path.insert(0, "src"); os.environ["PYTHONPATH"] = "src"
BACKEND = "vllm"

In [ ]:
# 1. Calibration on MATH-500 problems that are NOT in the test set (sets the threshold tau)
!CUDA_VISIBLE_DEVICES=0 python -m sr p2-calibrate --config configs/phase2_calib.json --backend $BACKEND
cal = json.load(open("runs/phase2_calib/calibration.json")); cal

**Check before the main run:** `share_answers_triggered` should be roughly 0.15 to 0.5;
`auroc_min_conf_predicts_wrong` above 0.5 means low confidence does flag wrong answers;
`accuracy_if_rule_would_fire` should be lower than `accuracy_if_rule_silent`.

In [ ]:
# 2. Main run: both GPUs in parallel, then merge
!(CUDA_VISIBLE_DEVICES=0 python -m sr p2-run --config configs/phase2_s0.json --backend $BACKEND > p2s0.log 2>&1 & \
  CUDA_VISIBLE_DEVICES=1 python -m sr p2-run --config configs/phase2_s1.json --backend $BACKEND > p2s1.log 2>&1 & \
  wait)
!tail -3 p2s0.log p2s1.log
!python -m sr merge --runs runs/phase2_s0 runs/phase2_s1 --out runs/phase2

In [ ]:
# 3. Analysis
from IPython.display import Image, display
from sr.phase2 import analyze2
res = analyze2("runs/phase2")
display(res["summary"].round(3)); display(res["effects"].round(3))
display(res["pairing"]); print(res["detection"])
display(Image("runs/phase2/analysis/injection_effects.png"))

In [ ]:
!zip -qr phase2_runs.zip runs/phase2_calib runs/phase2
print("download phase2_runs.zip from the Output panel")